# Build 02 — SFP Loop Detection Framework

> *A reusable `SFPDetector` class that runs a 4-step diagnostic on multi-version claims data*

**EN:** The centrepiece deliverable: a Python class an Allianz analyst could point at any claims dataframe (with multiple model versions) and get a structured "loop risk" report. The four steps test orthogonal aspects: temporal correlation, label generation mechanism, action–outcome confounding, and segment blind spots.

**KR:** 가장 중요한 산출물 중 하나. Allianz 분석가가 (모델 버전 여러 개인) 어떤 claims DataFrame에도 적용해 "loop risk" 리포트를 얻는 Python 클래스. 4단계는 서로 다른 측면 검정: 시계열 상관, 라벨 생성 메커니즘, action-outcome 교란, segment 블라인드 스팟.

## 1. Imports

**EN:** scipy.stats for correlation tests, sklearn for the Granger-style logistic regression in step 3.

**KR:** scipy.stats로 상관 검정, sklearn은 step 3의 Granger 스타일 로지스틱 회귀.

In [5]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

from typing import Optional

## 2. SFPDetector Class — All 4 Steps in One Cell

**EN:** The full `SFPDetector` class with all four methods (step1 through step4) plus `run_detection` and `print_report`. We define it as one cell so Python sees a single class definition.

> **Method overview:**
> - `step1_temporal_correlation`: Spearman corr between version scores. High ρ + score-driven investigation = loop reinforcement.
> - `step2_label_mechanism`: Tests if score predicts investigation (AUC > 0.8 = MNAR labels = biased evaluation).
> - `step3_action_outcome_confounding`: Compares fraud rates of high-vs-low-score investigated claims; Granger-style test on vN→v(N+1).
> - `step4_segment_blind_spots`: Finds segments where mean score is high but investigation rate is low.
> - `run_detection`: Runs all 4 → `loop_risk_score` (0–1) + severity HIGH/MEDIUM/LOW.

**KR:** `SFPDetector` 클래스 전체(step1~4 + `run_detection` + `print_report`)를 한 셀에 정의. Python이 단일 클래스 정의로 인식하도록.

> **메서드 개요:**
> - `step1_temporal_correlation`: 버전 점수 간 Spearman 상관. 높은 ρ + 점수 기반 조사 = 루프 강화.
> - `step2_label_mechanism`: 점수가 조사를 예측하는지(AUC > 0.8 = MNAR 라벨 = 편향된 평가).
> - `step3_action_outcome_confounding`: 고/저점수 조사 그룹의 fraud rate 비교; vN→v(N+1) Granger 스타일 검정.
> - `step4_segment_blind_spots`: 점수는 높지만 조사율이 낮은 segment 찾기.
> - `run_detection`: 4단계 모두 실행 → `loop_risk_score`(0~1) + 심각도 HIGH/MEDIUM/LOW.

In [6]:
class SFPDetector:
    """
    Detects self-fulfilling prophecy loops in ML-driven insurance claim data.

    Args:
        claims_df: DataFrame with claims + model scores + investigation decisions + fraud labels
        model_versions: List of version identifiers, e.g. ['v1', 'v2', 'v3']
        score_prefix: Column name prefix for model score columns (e.g. 'model_' → 'model_v1_score')
        investigation_col: Column indicating whether a claim was investigated (binary)
        fraud_label_col: Observed fraud label (NaN for uninvestigated claims)
        feature_cols: Claim feature columns (used for partial correlation controls)
    """

    def __init__(
        self,
        claims_df: pd.DataFrame,
        model_versions: list[str],
        score_prefix: str = 'model_',
        investigation_col: str = 'investigated',
        fraud_label_col: str = 'observed_fraud',
        feature_cols: Optional[list[str]] = None,
        true_fraud_col: Optional[str] = None,
    ):
        self.df = claims_df.copy()
        self.versions = model_versions
        self.score_cols = [f'{score_prefix}{v}_score' for v in model_versions]
        self.invest_col = investigation_col
        self.fraud_col = fraud_label_col
        self.feature_cols = feature_cols or []
        self.true_fraud_col = true_fraud_col

        self._validate_inputs()

    def _validate_inputs(self):
        missing = [c for c in self.score_cols if c not in self.df.columns]
        if missing:
            raise ValueError(f"Missing score columns: {missing}")
        if self.invest_col not in self.df.columns:
            raise ValueError(f"Missing investigation column: {self.invest_col}")

    # ─────────────────────────────────────────────────────────────
    # STEP 1: Temporal Prediction Correlation
    # ─────────────────────────────────────────────────────────────

    def step1_temporal_correlation(self) -> dict:
        """
        Measure Spearman correlation between consecutive model version scores.

        Interpretation:
            - High correlation (ρ > 0.85) is expected if models are similar
            - BUT: if correlation is high AND investigation rate is also score-driven,
              this suggests the loop is reinforcing predictions, not just similar data

        Returns:
            dict with correlation matrix and loop suspicion flags
        """
        corr_matrix = {}
        flags = []

        for i, v1 in enumerate(self.versions):
            for j, v2 in enumerate(self.versions):
                if i < j:
                    rho, pval = stats.spearmanr(
                        self.df[self.score_cols[i]],
                        self.df[self.score_cols[j]]
                    )
                    pair = f'{v1}→{v2}'
                    corr_matrix[pair] = {'spearman_rho': round(rho, 4), 'p_value': round(pval, 6)}

                    # Flag if consecutive versions have very high correlation
                    if j == i + 1 and rho > 0.85:
                        flags.append(
                            f"Versions {v1} and {v2} are highly correlated (ρ={rho:.3f}) — "
                            f"loop may be reinforcing scores"
                        )

        return {
            'step': 1,
            'name': 'Temporal Prediction Correlation',
            'correlation_matrix': corr_matrix,
            'flags': flags,
            'risk_signal': len(flags) > 0,
        }

    # ─────────────────────────────────────────────────────────────
    # STEP 2: Label Generation Mechanism Test
    # ─────────────────────────────────────────────────────────────

    def step2_label_mechanism(self) -> dict:
        """
        Test whether P(investigated=1) is strongly driven by model score.

        If investigation is determined by the model score (threshold policy),
        then fraud labels are MCAR-violated — they are missing NOT at random.

        Tests:
            1. Point-biserial correlation: model_score vs investigated
            2. Logistic regression AUC: can we predict investigated from score alone?
            3. Investigation rate by score decile
        """
        results_per_version = {}
        flags = []

        for v, score_col in zip(self.versions, self.score_cols):
            scores = self.df[score_col].values
            investigated = self.df[self.invest_col].values.astype(float)

            # Point-biserial correlation
            rho, pval = stats.pointbiserialr(investigated, scores)

            # AUC: can score predict investigation?
            auc = roc_auc_score(investigated, scores)

            # Investigation rate by decile
            decile_labels = pd.qcut(scores, 10, labels=False, duplicates='drop')
            invest_by_decile = (
                pd.Series(investigated).groupby(decile_labels).mean()
            ).to_dict()

            # Concentration: investigation in top 2 deciles
            top_decile_invest_rate = invest_by_decile.get(9, 0)
            bottom_decile_invest_rate = invest_by_decile.get(0, 0)
            concentration_ratio = (
                top_decile_invest_rate / bottom_decile_invest_rate
                if bottom_decile_invest_rate > 0 else float('inf')
            )

            results_per_version[v] = {
                'pointbiserial_rho': round(rho, 4),
                'investigation_auc': round(auc, 4),
                'concentration_ratio': round(concentration_ratio, 2),
                'invest_rate_top_decile': round(top_decile_invest_rate, 4),
                'invest_rate_bottom_decile': round(bottom_decile_invest_rate, 4),
            }

            if auc > 0.80:
                flags.append(
                    f"Version {v}: Model score predicts investigation with AUC={auc:.3f} — "
                    f"labels are NOT missing at random (MAR violation)"
                )
            if concentration_ratio > 10:
                flags.append(
                    f"Version {v}: Top decile investigated {concentration_ratio:.1f}× more than "
                    f"bottom decile — strong score-driven selection"
                )

        return {
            'step': 2,
            'name': 'Label Generation Mechanism Test',
            'results_per_version': results_per_version,
            'flags': flags,
            'risk_signal': len(flags) > 0,
        }

    # ─────────────────────────────────────────────────────────────
    # STEP 3: Action-Outcome Confounding Test
    # ─────────────────────────────────────────────────────────────

    def step3_action_outcome_confounding(self) -> dict:
        """
        Test whether investigation decision confounds the fraud label.

        Core test:
            Compare observed fraud rate in:
              (a) Model-driven investigated claims (score-selected)
              (b) Randomly investigated claims (if a random sample exists)

            If (a) >> (b): the investigation decision is a strong confounder
            → fraud labels are biased by the model's own selection

        Also computes:
            - Odds ratio: P(fraud | investigated) / P(fraud | random_sample)
            - Granger-style test: do vN scores predict v(N+1) LABELS after controlling for features?
        """
        flags = []

        # Check for random investigation sample (if available)
        # We use the first version's scores and assume claims in the bottom quartile
        # of the score were investigated "randomly" (they shouldn't have been)
        first_score_col = self.score_cols[0]
        scores_v1 = self.df[first_score_col].values
        investigated = self.df[self.invest_col].values

        fraud_labels = self.df[self.fraud_col].copy()
        investigated_mask = investigated == 1
        uninvestigated_mask = investigated == 0

        n_investigated = investigated_mask.sum()
        n_uninvestigated = uninvestigated_mask.sum()

        # Observed fraud rate in investigated claims
        investigated_fraud_labels = fraud_labels[investigated_mask].dropna()
        observed_fraud_rate = investigated_fraud_labels.mean() if len(investigated_fraud_labels) > 0 else np.nan

        # Compare: low-score investigated claims vs high-score investigated claims
        # (proxy for random vs model-driven)
        median_score = np.median(scores_v1[investigated_mask])
        low_score_invest_mask = investigated_mask & (scores_v1 < median_score)
        high_score_invest_mask = investigated_mask & (scores_v1 >= median_score)

        low_fraud_rate = fraud_labels[low_score_invest_mask].dropna().mean()
        high_fraud_rate = fraud_labels[high_score_invest_mask].dropna().mean()

        rate_ratio = high_fraud_rate / low_fraud_rate if low_fraud_rate > 0 else float('inf')

        if rate_ratio > 3:
            flags.append(
                f"Fraud rate in high-score investigated claims ({high_fraud_rate:.3f}) is "
                f"{rate_ratio:.1f}× higher than low-score claims ({low_fraud_rate:.3f}) — "
                f"investigation decision strongly confounds fraud labels"
            )

        # Granger-style test: does vN score predict v(N+1) labels (controlling for features)?
        granger_results = {}
        if len(self.versions) >= 2 and self.fraud_col in self.df.columns:
            for i in range(len(self.versions) - 1):
                v_curr = self.versions[i]
                v_next = self.versions[i + 1]
                score_col_curr = self.score_cols[i]
                invest_col_curr = f'model_{v_curr}_investigated' if f'model_{v_curr}_investigated' in self.df.columns else self.invest_col
                fraud_col_curr = f'model_{v_next}_observed_fraud' if f'model_{v_next}_observed_fraud' in self.df.columns else self.fraud_col

                if fraud_col_curr not in self.df.columns:
                    continue

                # Only look at claims investigated in v_next
                mask = self.df[invest_col_curr].fillna(0) == 1
                sub = self.df[mask].dropna(subset=[score_col_curr, fraud_col_curr])
                if len(sub) < 100:
                    continue

                y = sub[fraud_col_curr].astype(int)
                X_with_score = sub[[score_col_curr] + self.feature_cols].values
                X_without_score = sub[self.feature_cols].values if self.feature_cols else None

                try:
                    model_with = LogisticRegression(max_iter=500).fit(X_with_score, y)
                    auc_with = roc_auc_score(y, model_with.predict_proba(X_with_score)[:, 1])
                    granger_results[f'{v_curr}→{v_next}'] = {
                        'auc_with_prior_score': round(auc_with, 4),
                    }
                    if X_without_score is not None:
                        model_without = LogisticRegression(max_iter=500).fit(X_without_score, y)
                        auc_without = roc_auc_score(y, model_without.predict_proba(X_without_score)[:, 1])
                        incremental_auc = auc_with - auc_without
                        granger_results[f'{v_curr}→{v_next}']['auc_without_prior_score'] = round(auc_without, 4)
                        granger_results[f'{v_curr}→{v_next}']['incremental_auc_from_prior_score'] = round(incremental_auc, 4)
                        if incremental_auc > 0.03:
                            flags.append(
                                f"Version {v_curr} scores predict version {v_next} fraud labels "
                                f"with incremental AUC gain of {incremental_auc:.3f} — "
                                f"prior model is influencing future labels (loop signal)"
                            )
                except Exception:
                    pass

        return {
            'step': 3,
            'name': 'Action-Outcome Confounding Test',
            'n_investigated': int(n_investigated),
            'n_uninvestigated': int(n_uninvestigated),
            'observed_fraud_rate': round(observed_fraud_rate, 4) if not np.isnan(observed_fraud_rate) else None,
            'fraud_rate_high_score_investigated': round(high_fraud_rate, 4),
            'fraud_rate_low_score_investigated': round(low_fraud_rate, 4),
            'rate_ratio': round(rate_ratio, 2),
            'granger_style_results': granger_results,
            'flags': flags,
            'risk_signal': len(flags) > 0,
        }

    # ─────────────────────────────────────────────────────────────
    # STEP 4: Segment Blind Spot Analysis
    # ─────────────────────────────────────────────────────────────

    def step4_segment_blind_spots(
        self,
        segment_cols: Optional[list[str]] = None
    ) -> dict:
        """
        Identify claim segments with systematically low investigation rates
        relative to their estimated fraud risk (blind spots in the model).

        For each segment (e.g. postcode area, claim type, value band):
            - Compute average model score (proxy for estimated risk)
            - Compute actual investigation rate
            - Compute 'investigation gap': score rank - investigation rank
            - High gap = segment is rated risky but rarely investigated

        Returns ranked blind spot list with estimated missed fraud volume.
        """
        if segment_cols is None:
            # Auto-detect potential segment columns
            segment_cols = [
                c for c in self.df.columns
                if c not in self.score_cols
                and c != self.invest_col
                and c != self.fraud_col
                and self.df[c].dtype in ['object', 'category', 'int64']
                and self.df[c].nunique() < 50
            ]

        if not segment_cols:
            return {
                'step': 4,
                'name': 'Segment Blind Spot Analysis',
                'blind_spots': [],
                'flags': ['No suitable segment columns found — provide segment_cols parameter'],
                'risk_signal': False,
            }

        # Use last model version scores as the current "risk estimate"
        latest_score_col = self.score_cols[-1]
        scores = self.df[latest_score_col]
        investigated = self.df[self.invest_col]

        blind_spots = []

        for seg_col in segment_cols:
            try:
                group = self.df.groupby(seg_col).agg(
                    n_claims=(seg_col, 'count'),
                    mean_score=(latest_score_col, 'mean'),
                    investigation_rate=(self.invest_col, 'mean'),
                ).reset_index()

                group = group[group['n_claims'] >= 50]  # minimum segment size

                # Rank segments by score (high = risky) and by investigation rate
                group['score_rank'] = group['mean_score'].rank(ascending=False)
                group['invest_rank'] = group['investigation_rate'].rank(ascending=False)

                # Investigation gap: high positive = high risk but rarely investigated
                group['investigation_gap'] = group['score_rank'] - group['invest_rank']

                # Add observed fraud rate for investigated claims in this segment
                if self.fraud_col in self.df.columns:
                    invest_mask = self.df[self.invest_col] == 1
                    fraud_by_seg = (
                        self.df[invest_mask]
                        .groupby(seg_col)[self.fraud_col]
                        .mean()
                        .rename('observed_fraud_rate')
                        .reset_index()
                    )
                    group = group.merge(fraud_by_seg, on=seg_col, how='left')
                else:
                    group['observed_fraud_rate'] = np.nan

                group = group.sort_values('investigation_gap', ascending=False)
                top_blind_spots = group.head(5)

                for _, row in top_blind_spots.iterrows():
                    if row['investigation_gap'] > 3:
                        blind_spots.append({
                            'segment_col': seg_col,
                            'segment_value': row[seg_col],
                            'n_claims': int(row['n_claims']),
                            'mean_risk_score': round(row['mean_score'], 4),
                            'investigation_rate': round(row['investigation_rate'], 4),
                            'score_rank': int(row['score_rank']),
                            'invest_rank': int(row['invest_rank']),
                            'investigation_gap': round(row['investigation_gap'], 1),
                            'observed_fraud_rate': round(row.get('observed_fraud_rate', np.nan), 4),
                        })
            except Exception:
                continue

        blind_spots.sort(key=lambda x: x['investigation_gap'], reverse=True)
        flags = []
        if blind_spots:
            top = blind_spots[0]
            flags.append(
                f"Top blind spot: {top['segment_col']}={top['segment_value']} — "
                f"mean risk score {top['mean_risk_score']:.3f} but investigation rate only "
                f"{top['investigation_rate']:.1%}"
            )

        return {
            'step': 4,
            'name': 'Segment Blind Spot Analysis',
            'n_blind_spots_found': len(blind_spots),
            'blind_spots': blind_spots[:10],
            'flags': flags,
            'risk_signal': len(blind_spots) > 0,
        }

    # ─────────────────────────────────────────────────────────────
    # FULL DETECTION PIPELINE
    # ─────────────────────────────────────────────────────────────

    def run_detection(self, segment_cols: Optional[list[str]] = None) -> dict:
        """Run all four detection steps and return a complete loop risk report."""
        step1 = self.step1_temporal_correlation()
        step2 = self.step2_label_mechanism()
        step3 = self.step3_action_outcome_confounding()
        step4 = self.step4_segment_blind_spots(segment_cols=segment_cols)

        steps = [step1, step2, step3, step4]
        n_risk_signals = sum(s['risk_signal'] for s in steps)

        loop_risk_score = n_risk_signals / len(steps)  # 0–1 score
        severity = (
            'HIGH' if loop_risk_score >= 0.75 else
            'MEDIUM' if loop_risk_score >= 0.50 else
            'LOW'
        )

        all_flags = []
        for step in steps:
            all_flags.extend(step['flags'])

        return {
            'loop_risk_score': loop_risk_score,
            'severity': severity,
            'n_steps_flagged': n_risk_signals,
            'all_flags': all_flags,
            'step_results': {
                'step1_temporal_correlation': step1,
                'step2_label_mechanism': step2,
                'step3_action_confounding': step3,
                'step4_blind_spots': step4,
            }
        }

    def print_report(self, report: dict) -> None:
        """Pretty-print the detection report."""
        print("\n" + "=" * 65)
        print("  SFP Loop Detection Report")
        print("=" * 65)
        print(f"  Loop Risk Score : {report['loop_risk_score']:.0%}  [{report['severity']}]")
        print(f"  Steps Flagged   : {report['n_steps_flagged']} / 4")
        print()
        print("  Flags:")
        for i, flag in enumerate(report['all_flags'], 1):
            # Wrap long flags
            print(f"  {i:2d}. {flag[:100]}")
            if len(flag) > 100:
                print(f"       {flag[100:]}")
        print()
        print("  Blind Spots:")
        blind_spots = report['step_results']['step4_blind_spots']['blind_spots']
        if blind_spots:
            for bs in blind_spots[:5]:
                print(f"    → {bs['segment_col']}={bs['segment_value']} | "
                      f"risk={bs['mean_risk_score']:.3f} | "
                      f"investigated={bs['investigation_rate']:.1%} | "
                      f"gap={bs['investigation_gap']:.0f} ranks")
        else:
            print("    None detected (no segment columns provided or insufficient data)")
        print("=" * 65)

---

# 👁️ Section A — Hands-on Demo: Detect the Loop in Real Data

> *We load `datasets/real/coil2000_with_sfp_loop.csv` (the synthetic-SFP-augmented dataset) and let the detector loose on it.*

**EN:** This is the live test. The detector should fire HIGH severity because the data has a hand-crafted SFP loop in it. Every step prints diagnostic numbers as it runs.

**KR:** 라이브 테스트. 이 데이터에는 수작업으로 SFP 루프가 박혀 있으므로 HIGH severity가 나와야 정상. 각 step이 실행되면서 진단 수치를 출력.

## A.1 Step 1 — Load real data and inspect

**EN:** Load the COIL2000+SFP file and print which columns the detector will use.

**KR:** COIL2000+SFP 파일 로드, detector가 사용할 칼럼들을 print.

In [7]:
DATA_PATH = '/Users/yeon/work/future-dobby/2026-H1/allianz/test/datasets/real/coil2000_with_sfp_loop.csv'
df_real = pd.read_csv(DATA_PATH)

print(f"📦 Loaded: {df_real.shape[0]:,} rows × {df_real.shape[1]} columns")
print()

required = ['model_v1_score', 'model_v2_score', 'investigated',
            'observed_fraud', 'true_fraud']
print("🔍 Columns the detector needs:")
for col in required:
    present = '✓' if col in df_real.columns else '✗'
    extra = f"   missing={df_real[col].isna().sum():,}" if col in df_real.columns else ''
    print(f"  {present}  {col}{extra}")

print()
print("📊 Quick stats:")
print(f"  Investigation rate           : {df_real['investigated'].mean():.1%}")
print(f"  Observed fraud rate (invest) : {df_real.loc[df_real['investigated']==1, 'observed_fraud'].mean():.3f}")
print(f"  True fraud rate (all)        : {df_real['true_fraud'].mean():.3f}")

📦 Loaded: 9,822 rows × 96 columns

🔍 Columns the detector needs:
  ✓  model_v1_score   missing=0
  ✓  model_v2_score   missing=0
  ✓  investigated   missing=0
  ✓  observed_fraud   missing=7,365
  ✓  true_fraud   missing=0

📊 Quick stats:
  Investigation rate           : 25.0%
  Observed fraud rate (invest) : 0.256
  True fraud rate (all)        : 0.120


## A.2 Step 2 — Build a v3 score so we have 3 versions

**EN:** The file ships only v1 and v2 scores. To test the temporal-correlation step properly, we synthesise a v3 score = blend of v1+v2 + noise (same trick as Build 00's mapping).

**KR:** 파일에 v1, v2만 있어서 시계열 상관 step을 제대로 테스트하려면 v3가 필요. v1+v2+노이즈 blend로 합성(Build 00 매핑과 동일).

In [8]:
rng = np.random.default_rng(42)
n = len(df_real)
df_real['model_v3_score'] = (
    0.4 * df_real['model_v1_score']
    + 0.4 * df_real['model_v2_score']
    + 0.2 * rng.random(n)
).clip(0, 1).round(4)

print(f"✅ Synthesised model_v3_score (mean={df_real['model_v3_score'].mean():.3f},"
      f" std={df_real['model_v3_score'].std():.3f})")

✅ Synthesised model_v3_score (mean=0.388, std=0.147)


## A.3 Step 3 — Instantiate the detector

**EN:** Wire up the detector with column names from the real data. `feature_cols` will be used in step 3's Granger-style test.

**KR:** 실데이터 칼럼명으로 detector 연결. `feature_cols`는 step 3의 Granger 스타일 테스트에 사용.

In [9]:
# Pick feature columns that exist in the data
demographic_features = ['MOSTYPE', 'MGEMOMV', 'MGEMLEEF', 'MOPLHOOG', 'MINKM30']
demographic_features = [c for c in demographic_features if c in df_real.columns]

detector = SFPDetector(
    claims_df=df_real,
    model_versions=['v1', 'v2', 'v3'],
    score_prefix='model_',
    investigation_col='investigated',
    fraud_label_col='observed_fraud',
    feature_cols=demographic_features,
    true_fraud_col='true_fraud',
)
print("✅ Detector initialised")
print(f"   Versions: {detector.versions}")
print(f"   Score cols: {detector.score_cols}")
print(f"   Feature cols ({len(detector.feature_cols)}): {detector.feature_cols}")

✅ Detector initialised
   Versions: ['v1', 'v2', 'v3']
   Score cols: ['model_v1_score', 'model_v2_score', 'model_v3_score']
   Feature cols (5): ['MOSTYPE', 'MGEMOMV', 'MGEMLEEF', 'MOPLHOOG', 'MINKM30']


## A.4 Step 4 — Run each detection step individually (so you see what's measured)

### Step 1 — Temporal correlation

In [10]:
s1 = detector.step1_temporal_correlation()
print("Correlation matrix between versions:")
for pair, vals in s1['correlation_matrix'].items():
    print(f"  {pair}: ρ = {vals['spearman_rho']:.4f}  (p = {vals['p_value']:.2e})")
print(f"\nRisk signal: {s1['risk_signal']}")
for flag in s1['flags']:
    print(f"  ⚠️  {flag}")

Correlation matrix between versions:
  v1→v2: ρ = 0.5533  (p = 0.00e+00)
  v1→v3: ρ = 0.7896  (p = 0.00e+00)
  v2→v3: ρ = 0.7747  (p = 0.00e+00)

Risk signal: False


### Step 2 — Label generation mechanism

In [11]:
s2 = detector.step2_label_mechanism()
print("Per-version label-mechanism diagnostics:")
print(f"  {'Version':<8} {'rho':<10} {'AUC':<10} {'Concentration':<16}")
for v, r in s2['results_per_version'].items():
    print(f"  {v:<8} {r['pointbiserial_rho']:<10.4f} {r['investigation_auc']:<10.4f} "
          f"{r['concentration_ratio']:<16}")
print(f"\nRisk signal: {s2['risk_signal']}")
for flag in s2['flags']:
    print(f"  ⚠️  {flag}")

Per-version label-mechanism diagnostics:
  Version  rho        AUC        Concentration   
  v1       0.4268     0.7738     3.38            
  v2       0.8166     1.0000     inf             
  v3       0.6491     0.9080     185.0           

Risk signal: True
  ⚠️  Version v2: Model score predicts investigation with AUC=1.000 — labels are NOT missing at random (MAR violation)
  ⚠️  Version v2: Top decile investigated inf× more than bottom decile — strong score-driven selection
  ⚠️  Version v3: Model score predicts investigation with AUC=0.908 — labels are NOT missing at random (MAR violation)
  ⚠️  Version v3: Top decile investigated 185.0× more than bottom decile — strong score-driven selection


### Step 3 — Action-outcome confounding

In [12]:
s3 = detector.step3_action_outcome_confounding()
print(f"  Observed fraud rate                : {s3['observed_fraud_rate']}")
print(f"  Fraud rate in high-score-invest    : {s3['fraud_rate_high_score_investigated']}")
print(f"  Fraud rate in low-score-invest     : {s3['fraud_rate_low_score_investigated']}")
print(f"  Rate ratio (high / low)            : {s3['rate_ratio']}×")
print()
if s3['granger_style_results']:
    print("Granger-style results:")
    for pair, vals in s3['granger_style_results'].items():
        print(f"  {pair}: {vals}")
print(f"\nRisk signal: {s3['risk_signal']}")
for flag in s3['flags']:
    print(f"  ⚠️  {flag}")

  Observed fraud rate                : 0.2564
  Fraud rate in high-score-invest    : 0.3317
  Fraud rate in low-score-invest     : 0.1809
  Rate ratio (high / low)            : 1.83×

Granger-style results:
  v1→v2: {'auc_with_prior_score': 0.6308, 'auc_without_prior_score': 0.5641, 'incremental_auc_from_prior_score': 0.0668}
  v2→v3: {'auc_with_prior_score': 0.6489, 'auc_without_prior_score': 0.5367, 'incremental_auc_from_prior_score': 0.1122}

Risk signal: True
  ⚠️  Version v1 scores predict version v2 fraud labels with incremental AUC gain of 0.067 — prior model is influencing future labels (loop signal)
  ⚠️  Version v2 scores predict version v3 fraud labels with incremental AUC gain of 0.112 — prior model is influencing future labels (loop signal)


### Step 4 — Segment blind spots

In [13]:
s4 = detector.step4_segment_blind_spots(segment_cols=demographic_features)
print(f"  Blind spots found: {s4['n_blind_spots_found']}")
if s4['blind_spots']:
    print("\n  Top 5 blind spots:")
    for bs in s4['blind_spots'][:5]:
        print(f"    {bs['segment_col']}={bs['segment_value']}: "
              f"score={bs['mean_risk_score']:.3f}, invest={bs['investigation_rate']:.1%}, "
              f"gap={bs['investigation_gap']:.0f} ranks")
print(f"\nRisk signal: {s4['risk_signal']}")

  Blind spots found: 8

  Top 5 blind spots:
    MOSTYPE=7.0: score=0.381, invest=36.1%, gap=21 ranks
    MOSTYPE=25.0: score=0.376, invest=31.8%, gap=17 ranks
    MOSTYPE=6.0: score=0.376, invest=27.3%, gap=14 ranks
    MOSTYPE=26.0: score=0.384, invest=29.1%, gap=11 ranks
    MOSTYPE=1.0: score=0.333, invest=24.3%, gap=11 ranks

Risk signal: True


## A.5 Step 5 — Full report

**EN:** Run everything together and print the consolidated report. This is what an Allianz governance review would see.

**KR:** 전체를 한 번에 실행, 통합 리포트 출력. Allianz 거버넌스 리뷰에 제출할 형태.

In [14]:
report = detector.run_detection(segment_cols=demographic_features)
detector.print_report(report)


  SFP Loop Detection Report
  Loop Risk Score : 75%  [HIGH]
  Steps Flagged   : 3 / 4

  Flags:
   1. Version v2: Model score predicts investigation with AUC=1.000 — labels are NOT missing at random (MA
       R violation)
   2. Version v2: Top decile investigated inf× more than bottom decile — strong score-driven selection
   3. Version v3: Model score predicts investigation with AUC=0.908 — labels are NOT missing at random (MA
       R violation)
   4. Version v3: Top decile investigated 185.0× more than bottom decile — strong score-driven selection
   5. Version v1 scores predict version v2 fraud labels with incremental AUC gain of 0.067 — prior model i
       s influencing future labels (loop signal)
   6. Version v2 scores predict version v3 fraud labels with incremental AUC gain of 0.112 — prior model i
       s influencing future labels (loop signal)
   7. Top blind spot: MOSTYPE=7.0 — mean risk score 0.381 but investigation rate only 36.1%

  Blind Spots:
    → MOSTYPE=7.0 | r

## 🎯 What you should observe

**EN:**
- **Step 1**: very high correlation between consecutive versions (ρ > 0.85 typically) — fine on its own but suspicious in combination
- **Step 2**: AUC > 0.8 for investigated prediction → labels are MNAR → standard evaluation is biased
- **Step 3**: rate ratio > 3 → investigation choice strongly confounds fraud labels
- **Step 4**: at least a few segments where model says "risky" but policy says "skip"
- Final report: **HIGH** severity, all 4 steps flagged

**KR:**
- **Step 1**: 연속 버전 간 매우 높은 상관 (보통 ρ > 0.85) — 단독으로는 괜찮지만 다른 신호와 결합 시 의심
- **Step 2**: 조사 예측 AUC > 0.8 → 라벨이 MNAR → 표준 평가가 편향됨
- **Step 3**: rate ratio > 3 → 조사 선택이 fraud 라벨을 강하게 교란
- **Step 4**: 모델은 "위험"이라 하지만 정책은 "스킵"하는 segment 다수 존재
- 최종 리포트: **HIGH** 심각도, 4단계 모두 플래그